# Transjakarta Demand & Service Analysis

**Stakeholder (framing):** Transjakarta Operations & Planning Division

**Objective:** Analyze passenger transaction data to understand demand patterns across
corridors and time periods, identify potential capacity pressure and service-access
disparities, detect data-completeness anomalies, and produce evidence-based
recommendations for service planning.

**⚠️ Data disclosure:** Transaction records in this dataset are synthetically generated
(Faker library) over Transjakarta's real corridor/route/stop structure. Route geography
is real; individual passenger transactions are not. Findings below demonstrate analytical
methodology on realistic-shaped data, not verified operational conditions. This is stated
explicitly in the README and dashboard as well.

**Success criteria:** 3–5 evidence-backed findings translated into 2–3 actionable
recommendations, each clearly distinguishing what the data directly shows vs. what would
need real operational data (bus counts, headways, fare revenue) to validate.


## 1. Setup

In [1]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option('display.max_columns', 50)
plt.rcParams['figure.figsize'] = (10, 5)

con = duckdb.connect(database=':memory:')


## 2. Load data

In [2]:
con.execute("""
    CREATE OR REPLACE TABLE raw_transactions AS
    SELECT * FROM read_csv_auto('Transjakarta.csv')
""")

con.execute("SELECT COUNT(*) AS n_rows FROM raw_transactions").df()


,n_rows
0,37900


In [3]:
con.execute("DESCRIBE raw_transactions").df()


,column_name,column_type,null,key,default,extra
0,transID,VARCHAR,YES,None,None,None
1,payCardID,VARCHAR,YES,None,None,None
2,payCardBank,VARCHAR,YES,None,None,None
3,payCardName,VARCHAR,YES,None,None,None
4,payCardSex,VARCHAR,YES,None,None,None
5,payCardBirthDate,BIGINT,YES,None,None,None
6,corridorID,VARCHAR,YES,None,None,None
7,corridorName,VARCHAR,YES,None,None,None
8,direction,DOUBLE,YES,None,None,None
9,tapInStops,VARCHAR,YES,None,None,None


## 3. Data understanding

Quick pass before cleaning: row count, date coverage, and missingness by column,
so cleaning decisions are based on actual data shape rather than assumption.


In [4]:
schema = con.execute("PRAGMA table_info('raw_transactions')").df()
cols = schema['name'].tolist()

missing_expr = ", ".join([f"ROUND(100.0 * SUM(CASE WHEN \"{c}\" IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS \"{c}\"" for c in cols])
missing_pct = con.execute(f"SELECT {missing_expr} FROM raw_transactions").df().T
missing_pct.columns = ['pct_missing']
missing_pct.sort_values('pct_missing', ascending=False)


,pct_missing
tapOutStops,6.04
corridorName,5.09
tapOutStopsLat,3.55
stopEndSeq,3.55
tapOutTime,3.55
tapOutStopsLon,3.55
tapOutStopsName,3.55
corridorID,3.32
tapInStops,3.20
payAmount,2.66


In [5]:
con.execute("""
    SELECT MIN(tapInTime) AS earliest_tap_in, MAX(tapInTime) AS latest_tap_in
    FROM raw_transactions
""").df()


,earliest_tap_in,latest_tap_in
0,2023-04-01 06:22:27,2023-04-30 21:55:41


## 4. Cleaning & feature engineering

- Parse timestamps
- Derive trip duration (minutes)
- Derive passenger age from birth year
- Flag peak hours (morning 05:00–09:00, evening 16:00–21:00 — based on known TJ ridership patterns)
- Flag weekday vs weekend
- Flag incomplete records (missing tap-out / corridor / stop) as a **data-completeness**
  signal rather than searching for duration outliers (duration field is artificially
  bounded 15–180 min in this dataset — see disclosure above, no organic outliers exist)


In [6]:
df = con.execute("SELECT * FROM raw_transactions").df()

df['tapInTime'] = pd.to_datetime(df['tapInTime'], errors='coerce')
df['tapOutTime'] = pd.to_datetime(df['tapOutTime'], errors='coerce')

df['trip_duration_min'] = (df['tapOutTime'] - df['tapInTime']).dt.total_seconds() / 60
df['tap_in_hour'] = df['tapInTime'].dt.hour
df['tap_in_date'] = df['tapInTime'].dt.date
df['day_of_week'] = df['tapInTime'].dt.day_name()
df['is_weekend'] = df['tapInTime'].dt.dayofweek >= 5

df['passenger_age'] = 2023 - df['payCardBirthDate']

def age_group(age):
    if pd.isna(age):
        return 'Unknown'
    if age < 20:
        return 'Youth (<20)'
    if age < 60:
        return 'Adult (20-59)'
    return 'Senior (60+)'

df['age_group'] = df['passenger_age'].apply(age_group)

def peak_flag(hour):
    if pd.isna(hour):
        return 'Unknown'
    if 5 <= hour <= 9:
        return 'Morning Peak'
    if 16 <= hour <= 21:
        return 'Evening Peak'
    return 'Off-Peak'

df['peak_period'] = df['tap_in_hour'].apply(peak_flag)

df['is_incomplete_record'] = (
    df['tapOutStops'].isna() | df['corridorName'].isna() | df['tapOutTime'].isna()
)

con.register('df_clean', df)
con.execute("CREATE OR REPLACE TABLE transactions AS SELECT * FROM df_clean")

df[['trip_duration_min', 'passenger_age', 'age_group', 'peak_period', 'is_incomplete_record']].describe(include='all')


,trip_duration_min,passenger_age,age_group,peak_period,is_incomplete_record
count,36556.000000,37900.000000,37900,37900,37900
unique,NaN,NaN,3,3,2
top,NaN,NaN,Adult (20-59),Evening Peak,False
freq,NaN,NaN,30089,18292,33805
mean,72.125424,32.910686,NaN,NaN,NaN
std,28.072912,13.051482,NaN,NaN,NaN
min,15.000000,11.000000,NaN,NaN,NaN
25%,51.133333,22.000000,NaN,NaN,NaN
50%,71.833333,33.000000,NaN,NaN,NaN
75%,95.800000,41.000000,NaN,NaN,NaN


---
## 5. Demand patterns
*Q1–Q4: Where/when is demand highest, trend over the month, capacity-pressure signals, weekday vs weekend*


In [7]:
# Q1: demand by corridor, day of week, hour


In [8]:
# Q2: demand trend over the observed period (daily volume line chart)


In [9]:
# Q3: corridors with demand concentration that may indicate capacity pressure (top-N + distribution shape)


In [10]:
# Q4: weekday vs weekend demand comparison


## 6. Service & access
*Q5–Q8: trip duration variation, geographic access gaps, payment-method patterns, demographic usage patterns*


In [11]:
# Q5: trip duration by corridor / stop / time of day


In [12]:
# Q6: geographic demand map (corridor/stop coordinates) — flagged for later cross-check against population density


In [13]:
# Q7: payment method (payCardBank) mix by corridor


In [14]:
# Q8: age_group / gender usage patterns by corridor


## 7. Data quality / completeness
*Q9–Q10: incomplete-record patterns as an operational signal, not duration-outlier detection*


In [15]:
# Q9: incomplete record rate by corridor / stop / time period


In [16]:
# Q10: are incomplete records concentrated in specific corridors or time windows (potential tap system reliability signal)?


## 8. Recommendations
*Q11–Q12: proposed service adjustments + what would need real operational data to validate; which findings are actionable vs. hypotheses*


In [17]:
# Q11 / Q12: synthesize findings above into 2-3 recommendations with explicit validation caveats
